In [78]:
from geneticengine.grammar.decorators import abstract
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.random.sources import NativeRandomSource
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.problems import SingleObjectiveProblem
from geneticengine.evaluation.budget import EvaluationBudget, TimeBudget
from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.problems import MultiObjectiveProblem

from analysis_helpers import calculate_expression_complexity

import time

from abc import ABC
from dataclasses import dataclass

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils import resample
from sklearn import metrics

from typing import Annotated
import pandas as pd

import re

In [79]:
dataset = pd.read_csv('../../datasets/parkinsons.csv')
X_df = dataset.drop(columns=['name', 'status'])
feature_names = X_df.columns
X = X_df.values
y = dataset['status'].values

In [80]:
n_features = X.shape[1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [81]:
weight_config = {
    "w_operator" : 0.3,
    "w_column": 0.2,
    "w_depth": 0.5
}

In [82]:
#core classes
class Feature(ABC):
    def evaluate(self, X):
        pass

@dataclass
class PrimitiveFeature(Feature):
    column_index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X):
        return X[:, self.column_index]
    
    def __str__(self):
        name = feature_names[self.column_index]
        safe = re.sub(r'[^A-Za-z0-9_]+', '_', str(name))
        return f"column_{safe}"
    
@dataclass
class Add(Feature):
    left : Feature #type feature
    right : Feature #type feature
    def evaluate(self, X):
        return self.left.evaluate(X) + self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} + {self.right})"

@dataclass
class Subtract(Feature):
    left : Feature
    right : Feature
    def evaluate(self, X):
        return self.left.evaluate(X) - self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} - {self.right})"


In [83]:
#grammar
components = [
    PrimitiveFeature,
    Add,
    Subtract
]

grammar = extract_grammar(components, Feature)
print(grammar)

Grammar<Starting=Feature,Productions={
Feature -> PrimitiveFeature(column_index: Annotated[int])|
	Add(left: Feature, right: Feature)|
	Subtract(left: Feature, right: Feature)
}


In [84]:
#fitness function
def fitness_function(feature: Feature) -> list[float]:
    start = time.perf_counter()

    feature_values = feature.evaluate(X_train)

    X_combined = np.hstack([X_train, feature_values.reshape(-1,1)])
    
    clf = make_pipeline(StandardScaler(), LogisticRegression(random_state=0, max_iter=200, solver='liblinear'))
    f1 = cross_val_score(clf, X_combined, y_train, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42), scoring='f1_macro').mean()

    complexity = complexity_function(feature, weight_config)

    elapsed = time.perf_counter() - start

    return [float(f1), float(complexity), float(elapsed)]

#complexity function (cuidado com o python limit recursion)
def complexity_function(feature: Feature, weight_config: dict) -> float:
    unique_features = set()
    def walk(node):
        if isinstance(node, PrimitiveFeature):
            unique_features.add(node.column_index)
            return 0,0 #operators, depth
        elif isinstance(node, Add) or isinstance(node, Subtract):
            left_operators, left_depth = walk(node.left)
            right_operators, right_depth = walk(node.right)
            return left_operators + right_operators + 1, max(left_depth, right_depth) + 1
        else:
            return 0,0 #unexpected node
    number_operators, depth = walk(feature)
    number_unique_features = len(unique_features)
    return (weight_config["w_operator"] * number_operators +
            weight_config["w_column"] * number_unique_features +
            weight_config["w_depth"] * depth)



In [85]:
#GP setup
rnd = NativeRandomSource(123)
decider = MaxDepthDecider(rnd, grammar, max_depth=5)
representation = TreeBasedRepresentation(grammar, decider)
objective = MultiObjectiveProblem(fitness_function=fitness_function, minimize=[False, True, True])

gp = GeneticProgramming(
    problem=objective,
    budget=TimeBudget(30),
    representation=representation,
    random=rnd,
    tracker= ProgressTracker(
        objective,
        recorders=[CSVSearchRecorder(csv_path='../../gp_outputs/tests.csv', problem=objective, fields={"Eval Time": lambda t,i,p:i.get_fitness(p).fitness_components[2],
                                                                                                       "Expression": lambda t, i, p: i.get_phenotype(),
                                                                                                       "F1 Score": lambda t, i, p: i.get_fitness(p).fitness_components[0],
                                                                                                       "Complexity": lambda t, i, p: i.get_fitness(p).fitness_components[1],
                                                                                                       }, only_record_best_individuals=True)]
    ),
    population_size=50,
)

solutions = gp.search()


In [86]:
best = max(solutions, key=lambda row: row.get_fitness(objective).fitness_components[0])

In [87]:
print(best.get_fitness(objective).fitness_components)

[0.7740714695932087, 1.2, 0.0357304000062868]


In [88]:
feat = best.get_phenotype()
print(feat)

(column_MDVP_Jitter_ - column_MDVP_RAP)


In [89]:
train_col = feat.evaluate(X_train).reshape(-1,1)
test_col = feat.evaluate(X_test).reshape(-1,1)
X_train_aug = np.hstack([X_train, train_col])
X_test_aug = np.hstack([X_test, test_col])

In [90]:
clf = make_pipeline(StandardScaler(), LogisticRegression(random_state=0, max_iter=200, solver='liblinear'))
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_f1 = cross_val_score(clf, X_train_aug, y_train, cv=cv, scoring='f1_macro').mean()
clf.fit(X_train_aug, y_train)
y_pred = clf.predict(X_test_aug)
test_f1 = metrics.f1_score(y_test, y_pred, average='macro')
print(f"Cross-validated F1 on training set (with new feature): {cv_f1}")
print(f"F1 on test set (with new feature): {test_f1}")


Cross-validated F1 on training set (with new feature): 0.7740714695932087
F1 on test set (with new feature): 0.8956289027653881


In [91]:
clf_base = make_pipeline(StandardScaler(), LogisticRegression(random_state=0, max_iter=200, solver='liblinear'))
base_cv = cross_val_score(clf_base, X_train, y_train, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42), scoring='f1_macro').mean()
clf_base.fit(X_train, y_train)
base_test = metrics.f1_score(y_test, clf_base.predict(X_test), average='macro')
print(f"Cross-validated F1 on training set (base): {base_cv}")
print(f"F1 on test set (base): {base_test}")

Cross-validated F1 on training set (base): 0.7473741451784929
F1 on test set (base): 0.8956289027653881


In [92]:
# print(f"{alg.get_fitness(gp.get_problem())}\n{alg.get_phenotype()}") #f1, expression complexity, evaluation time